## 0.提取文章

In [1]:
import os
import json
import xml.etree.ElementTree as ET

# 路径：notebook在 notebooks/ 下运行，所以用 ../ 回到项目根目录
xml_dir = "../data/raw/PMC000xxxxxx"
output_path = "../data/processed/dataset.json"

files = [f for f in os.listdir(xml_dir) if f.endswith(".xml")]
print(f"共发现 {len(files)} 个XML文件")

records = []

for fname in files:
    path = os.path.join(xml_dir, fname)
    try:
        tree = ET.parse(path)
        root = tree.getroot()

        title_el = root.find(".//article-title")
        title = "".join(title_el.itertext()).strip() if title_el is not None else None

        abstract_el = root.find(".//abstract")
        abstract = " ".join(abstract_el.itertext()).strip() if abstract_el is not None else None

        journal_el = root.find(".//journal-meta//journal-title")
        journal = journal_el.text.strip() if journal_el is not None and journal_el.text else None

        pmid_el = root.find('.//article-id[@pub-id-type="pmid"]')
        pmid = pmid_el.text.strip() if pmid_el is not None and pmid_el.text else None

        pub_date = None
        for pub_type in ["ppub", "epub", "pmc-release"]:
            date_el = root.find(f'.//pub-date[@pub-type="{pub_type}"]')
            if date_el is not None:
                year = date_el.findtext("year")
                month = date_el.findtext("month")
                day = date_el.findtext("day")
                if year:
                    pub_date = "-".join(filter(None, [year, month, day]))
                    break

        records.append({
            "pmc_id": fname.replace(".xml", ""),
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "journal": journal,
            "pub_date": pub_date,
        })
    except Exception as e:
        records.append({
            "pmc_id": fname.replace(".xml", ""),
            "pmid": None, "title": None, "abstract": None,
            "journal": None, "pub_date": None,
            "_parse_error": str(e),
        })

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=1)

print(f"提取完成，共 {len(records)} 条记录，已保存到 {output_path}")
print("样例:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

共发现 3028 个XML文件
提取完成，共 3028 条记录，已保存到 ../data/processed/dataset.json
样例:
{
  "pmc_id": "PMC524367",
  "pmid": "15462679",
  "title": "Perceived personal, social and environmental barriers to weight maintenance among young women: A community survey",
  "abstract": "Background Young women are a group at high risk of weight gain. This study examined a range of perceived personal, social and environmental barriers to physical activity and healthy eating for weight maintenance among young women, and how these varied by socioeconomic status (SES), overweight status and domestic situation. Methods In October-December 2001, a total of 445 women aged 18–32 years, selected randomly from the Australian electoral roll, completed a mailed self-report survey that included questions on 11 barriers to physical activity and 11 barriers to healthy eating (relating to personal, social and environmental factors). Height, weight and socio-demographic details were also obtained. Statistical analyses were con

In [49]:
import os
import json
import xml.etree.ElementTree as ET

# 改动点：从"单一目录"改成"目录列表"，遍历所有已下载的分片
xml_dirs = [
    "../data/raw/PMC000xxxxxx",
    "../data/raw/PMC001xxxxxx",
    "../data/raw/PMC002xxxxxx",
]
output_path = "../data/processed/dataset_expanded.json"  # 新文件名，不覆盖旧的3028篇版本

# 汇总所有分片目录下的xml文件路径，记录来源分片方便排查问题
all_files = []
for d in xml_dirs:
    fnames = [f for f in os.listdir(d) if f.endswith(".xml")]
    print(f"{d}: 发现 {len(fnames)} 个XML文件")
    all_files.extend([(d, f) for f in fnames])

print(f"共发现 {len(all_files)} 个XML文件（{len(xml_dirs)}个分片合计）")

records = []
for xml_dir, fname in all_files:
    path = os.path.join(xml_dir, fname)
    try:
        tree = ET.parse(path)
        root = tree.getroot()
        title_el = root.find(".//article-title")
        title = "".join(title_el.itertext()).strip() if title_el is not None else None
        abstract_el = root.find(".//abstract")
        abstract = " ".join(abstract_el.itertext()).strip() if abstract_el is not None else None
        journal_el = root.find(".//journal-meta//journal-title")
        journal = journal_el.text.strip() if journal_el is not None and journal_el.text else None
        pmid_el = root.find('.//article-id[@pub-id-type="pmid"]')
        pmid = pmid_el.text.strip() if pmid_el is not None and pmid_el.text else None
        pub_date = None
        for pub_type in ["ppub", "epub", "pmc-release"]:
            date_el = root.find(f'.//pub-date[@pub-type="{pub_type}"]')
            if date_el is not None:
                year = date_el.findtext("year")
                month = date_el.findtext("month")
                day = date_el.findtext("day")
                if year:
                    pub_date = "-".join(filter(None, [year, month, day]))
                    break
        records.append({
            "pmc_id": fname.replace(".xml", ""),
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "journal": journal,
            "pub_date": pub_date,
        })
    except Exception as e:
        records.append({
            "pmc_id": fname.replace(".xml", ""),
            "pmid": None, "title": None, "abstract": None,
            "journal": None, "pub_date": None,
            "_parse_error": str(e),
        })

# 去重：理论上不同分片目录之间pmc_id不会重复，但保险起见按pmc_id去重一次
before_dedup = len(records)
seen = set()
deduped = []
for r in records:
    if r["pmc_id"] not in seen:
        seen.add(r["pmc_id"])
        deduped.append(r)
records = deduped
if before_dedup != len(records):
    print(f"去重：{before_dedup} -> {len(records)}（发现{before_dedup - len(records)}条重复pmc_id）")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=1)

parse_errors = sum(1 for r in records if r.get("_parse_error"))
print(f"提取完成，共 {len(records)} 条记录（解析失败 {parse_errors} 条），已保存到 {output_path}")
print("样例:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

../data/raw/PMC000xxxxxx: 发现 3028 个XML文件
../data/raw/PMC001xxxxxx: 发现 27518 个XML文件
../data/raw/PMC002xxxxxx: 发现 122575 个XML文件
共发现 153121 个XML文件（3个分片合计）
提取完成，共 153121 条记录（解析失败 0 条），已保存到 ../data/processed/dataset_expanded.json
样例:
{
  "pmc_id": "PMC524367",
  "pmid": "15462679",
  "title": "Perceived personal, social and environmental barriers to weight maintenance among young women: A community survey",
  "abstract": "Background Young women are a group at high risk of weight gain. This study examined a range of perceived personal, social and environmental barriers to physical activity and healthy eating for weight maintenance among young women, and how these varied by socioeconomic status (SES), overweight status and domestic situation. Methods In October-December 2001, a total of 445 women aged 18–32 years, selected randomly from the Australian electoral roll, completed a mailed self-report survey that included questions on 11 barriers to physical activity and 11 barriers to healthy ea

## 1.检查字段完整性

In [51]:
import json

with open("../data/processed/dataset.json", encoding="utf-8") as f:
    data = json.load(f)

n = len(data)
print(f"=== 数据集总量: {n} 条 ===\n")

print("--- 字段缺失率 ---")
fields = ["pmid", "title", "abstract", "journal", "pub_date"]
missing_report = {}
for field in fields:
    missing = sum(1 for r in data if not r.get(field))
    rate = missing / n * 100
    missing_report[field] = rate
    print(f"{field:12s}: 缺失 {missing:4d} 条 / {n} ({rate:.2f}%)")

print()
abstract_missing_rate = missing_report["abstract"]
if abstract_missing_rate > 1:
    print(f"[判断] abstract 缺失率 {abstract_missing_rate:.2f}% > 1%，需要制定清洗策略。")
    print("[策略建议] abstract 是RAG检索的核心内容，缺失时无法靠填充弥补，")
    print("           建议直接丢弃这部分记录。")
else:
    print(f"[判断] abstract 缺失率 {abstract_missing_rate:.2f}% <= 1%，可直接丢弃，对数据量影响很小。")

# 解析失败（XML格式异常）导致的缺失
parse_errors = sum(1 for r in data if r.get("_parse_error"))
print(f"\n其中因XML解析报错导致整条记录缺失: {parse_errors} 条")

=== 数据集总量: 3028 条 ===

--- 字段缺失率 ---
pmid        : 缺失  275 条 / 3028 (9.08%)
title       : 缺失    0 条 / 3028 (0.00%)
abstract    : 缺失  253 条 / 3028 (8.36%)
journal     : 缺失    0 条 / 3028 (0.00%)
pub_date    : 缺失   12 条 / 3028 (0.40%)

[判断] abstract 缺失率 8.36% > 1%，需要制定清洗策略。
[策略建议] abstract 是RAG检索的核心内容，缺失时无法靠填充弥补，
           建议直接丢弃这部分记录。

其中因XML解析报错导致整条记录缺失: 0 条


In [53]:
import pandas as pd
import json

# ---------- 加载扩充后的数据集 ----------
with open("../data/processed/dataset_expanded.json", encoding="utf-8") as f:
    data_expanded = json.load(f)

df_raw_expanded = pd.DataFrame(data_expanded)
print(f"原始记录数: {len(df_raw_expanded)}")

# ---------- 字段完整性检查 ----------
fields_to_check = ["pmid", "title", "abstract", "journal", "pub_date"]
missing_stats = []
for field in fields_to_check:
    missing_count = df_raw_expanded[field].isna().sum() + (df_raw_expanded[field] == "").sum()
    missing_rate = missing_count / len(df_raw_expanded)
    missing_stats.append({"字段": field, "缺失数": missing_count, "缺失率": f"{missing_rate:.2%}"})

missing_df = pd.DataFrame(missing_stats)
print(missing_df.to_string(index=False))

# ---------- 清洗决策：abstract缺失率是否超过1%阈值 ----------
abstract_missing_rate = (df_raw_expanded["abstract"].isna().sum() + (df_raw_expanded["abstract"] == "").sum()) / len(df_raw_expanded)
print(f"\nabstract缺失率: {abstract_missing_rate:.2%}")
if abstract_missing_rate > 0.01:
    print("超过1%阈值，按原策略丢弃abstract缺失的记录")
else:
    print("未超过1%阈值")

df_clean_expanded = df_raw_expanded[
    df_raw_expanded["abstract"].notna() & (df_raw_expanded["abstract"].str.strip() != "")
].copy()
print(f"\n清洗后有效记录数: {len(df_clean_expanded)}（丢弃了 {len(df_raw_expanded) - len(df_clean_expanded)} 条abstract缺失的记录）")

原始记录数: 153121
      字段   缺失数   缺失率
    pmid  5570 3.64%
   title    20 0.01%
abstract 14632 9.56%
 journal     0 0.00%
pub_date    12 0.01%

abstract缺失率: 9.56%
超过1%阈值，按原策略丢弃abstract缺失的记录

清洗后有效记录数: 138489（丢弃了 14632 条abstract缺失的记录）


## 2.基础质量分析

In [55]:
import re

valid = [r for r in data if r.get("abstract")]
print(f"有效记录数（abstract非空）: {len(valid)}\n")

# 超短摘要
short_threshold = 30 
short_abstracts = [r for r in valid if len(r["abstract"].split()) < short_threshold]
print(f"超短摘要 (<{short_threshold} 词): {len(short_abstracts)} 条 ({len(short_abstracts)/len(valid)*100:.2f}%)")
if short_abstracts:
    print(f"  示例: {short_abstracts[0]['abstract'][:150]}")
    print(f"  来源: {short_abstracts[0]['pmc_id']}")

# 编码错误检测
def has_encoding_error(text):
    return bool(re.search(r'[\x80-\x9f]|\ufffd', text))

encoding_errors = [r for r in valid if has_encoding_error(r["abstract"])]
print(f"\n疑似编码错误: {len(encoding_errors)} 条 ({len(encoding_errors)/len(valid)*100:.2f}%)")
if encoding_errors:
    print(f"  示例: {repr(encoding_errors[0]['abstract'][:100])}")
    print(f"  来源: {encoding_errors[0]['pmc_id']}")

有效记录数（abstract非空）: 2775

超短摘要 (<30 词): 200 条 (7.21%)
  示例: A  Drosophila  protein-protein interaction map was constructed using the LexA system, complementing a previous map using the GAL4 system and adding ma
  来源: PMC545799

疑似编码错误: 0 条 (0.00%)


In [57]:
# ---------- 超短摘要检查（<30词）----------
short_abstract_count = df_clean_expanded["abstract"].apply(lambda x: len(x.split()) < 30).sum()
short_abstract_rate = short_abstract_count / len(df_clean_expanded)
print(f"超短摘要（<30词）: {short_abstract_count}篇，占有效记录 {short_abstract_rate:.2%}")

# ---------- 编码错误检查（简单启发式：出现常见乱码字符）----------
def has_encoding_issue(text):
    # 常见编码错误标志：出现替换字符、连续问号、或者常见mojibake模式
    suspicious_chars = ["�", "Ã", "â€"]
    return any(c in text for c in suspicious_chars)

encoding_issue_count = df_clean_expanded["abstract"].apply(has_encoding_issue).sum()
encoding_issue_rate = encoding_issue_count / len(df_clean_expanded)
print(f"疑似编码错误: {encoding_issue_count}篇，占有效记录 {encoding_issue_rate:.2%}")

超短摘要（<30词）: 3608篇，占有效记录 2.61%
疑似编码错误: 5篇，占有效记录 0.00%


## 3.关键字段分析

In [59]:
from collections import Counter

journals = Counter(r["journal"] for r in valid if r.get("journal"))
print(f"journal 唯一值数量: {len(journals)}")
print("出现最多的5个期刊:", journals.most_common(5))

years = Counter(r["pub_date"][:4] for r in valid if r.get("pub_date"))
print(f"\npub_date 年份跨度: {min(years)} - {max(years)}")
print("各年份文章数(部分，按年份排序前5):", dict(sorted(years.items())[:5]))

pmid_present = sum(1 for r in valid if r.get("pmid"))
print(f"\n有效记录中pmid存在的比例: {pmid_present}/{len(valid)} ({pmid_present/len(valid)*100:.2f}%)")

sample_pmid = next(r["pmid"] for r in valid if r.get("pmid"))
print(f"pmid格式示例: {sample_pmid}，是否纯数字: {sample_pmid.isdigit()}")

journal 唯一值数量: 139
出现最多的5个期刊: [('PLoS Biology', 349), ('BMC Bioinformatics', 161), ('PLoS Medicine', 90), ('BMC Cancer', 88), ('BMC Genomics', 86)]

pub_date 年份跨度: 2003 - 2024
各年份文章数(部分，按年份排序前5): {'2003': 21, '2004': 1850, '2005': 869, '2024': 23}

有效记录中pmid存在的比例: 2735/2775 (98.56%)
pmid格式示例: 15462679，是否纯数字: True


In [61]:
# ---------- journal 唯一值数量 ----------
journal_unique = df_clean_expanded["journal"].nunique()
print(f"journal唯一值数量: {journal_unique}")

# ---------- pub_date 年份分布 ----------
df_clean_expanded["pub_year"] = df_clean_expanded["pub_date"].str.extract(r"(\d{4})").astype(float)
year_counts = df_clean_expanded["pub_year"].value_counts().sort_index()
print(f"\npub_date年份跨度: {int(year_counts.index.min())} - {int(year_counts.index.max())}")
print(f"\n年份分布（前10个占比最高的年份）:")
year_dist = (df_clean_expanded["pub_year"].value_counts(normalize=True) * 100).sort_values(ascending=False).head(10)
print(year_dist.round(2))

# ---------- pmid 完整率 ----------
pmid_present_rate = df_clean_expanded["pmid"].notna().sum() / len(df_clean_expanded)
print(f"\npmid存在比例: {pmid_present_rate:.2%}")

journal唯一值数量: 1186

pub_date年份跨度: 1947 - 2024

年份分布（前10个占比最高的年份）:
pub_year
2009.0    23.22
2010.0    22.90
2008.0    17.78
2007.0    10.35
2006.0     7.24
2005.0     4.89
2004.0     2.22
2003.0     0.86
1999.0     0.83
2002.0     0.77
Name: proportion, dtype: float64

pmid存在比例: 99.62%


## 4.Token长度分析

In [63]:
try:
    import tiktoken
    print("tiktoken 可用")
except ImportError:
    print("tiktoken 未安装")

try:
    import transformers
    print("transformers 可用，版本:", transformers.__version__)
except ImportError:
    print("transformers 未安装")

tiktoken 未安装
transformers 可用，版本: 5.12.1


In [65]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")

lengths = []
for r in valid:
    full_text = f"{r.get('title') or ''}\n{r['abstract']}"
    n_tokens = len(tokenizer.encode(full_text, truncation=False))
    lengths.append(n_tokens)

lengths.sort()
n = len(lengths)

def percentile(p):
    idx = int(n * p / 100)
    idx = min(idx, n - 1)
    return lengths[idx]

print(f"=== Token长度分布（{n}篇，title+abstract拼接，真实bge tokenizer）===\n")
for p in [5, 25, 50, 75, 90, 95, 99, 100]:
    print(f"  P{p:<3d}: {percentile(p):4d} tokens")

mean_len = sum(lengths) / n
print(f"\n  均值: {mean_len:.0f} tokens")
print(f"  最小: {lengths[0]} / 最大: {lengths[-1]}")

embedding_limit = 512
p95 = percentile(95)
p99 = percentile(99)
over_limit = sum(1 for l in lengths if l > embedding_limit)

print(f"\n=== 与bge-base-en-v1.5上限({embedding_limit} tokens)对比 ===")
print(f"  P95 = {p95} tokens")
print(f"  P99 = {p99} tokens")
print(f"  超过{embedding_limit} tokens的文章占比: {over_limit}/{n} ({over_limit/n*100:.2f}%)")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (535 > 512). Running this sequence through the model will result in indexing errors


=== Token长度分布（2775篇，title+abstract拼接，真实bge tokenizer）===

  P5  :   44 tokens
  P25 :  261 tokens
  P50 :  352 tokens
  P75 :  440 tokens
  P90 :  527 tokens
  P95 :  576 tokens
  P99 :  686 tokens
  P100:  952 tokens

  均值: 343 tokens
  最小: 8 / 最大: 952

=== 与bge-base-en-v1.5上限(512 tokens)对比 ===
  P95 = 576 tokens
  P99 = 686 tokens
  超过512 tokens的文章占比: 328/2775 (11.82%)


In [67]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")

def count_tokens(text):
    return len(tokenizer.encode(text, truncation=False))

# 抽样2万篇做token统计（全量13.8万篇跑一遍太慢，抽样足够反映分布）
sample_df = df_clean_expanded.sample(n=20000, random_state=42)

def build_full_text(row):
    title = (row["title"] or "").rstrip(".")
    return f"{title}. {row['abstract']}"

sample_df = sample_df.copy()
sample_df["token_count"] = sample_df.apply(lambda r: count_tokens(build_full_text(r)), axis=1)

print("Token数分位数统计（抽样2万篇）:")
print(sample_df["token_count"].describe())
print(f"\nP90: {sample_df['token_count'].quantile(0.90):.0f}")
print(f"P95: {sample_df['token_count'].quantile(0.95):.0f}")
print(f"P99: {sample_df['token_count'].quantile(0.99):.0f}")

over_512_rate = (sample_df["token_count"] > 512).mean()
print(f"\n超过512 tokens的文章占比: {over_512_rate:.2%}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (567 > 512). Running this sequence through the model will result in indexing errors


Token数分位数统计（抽样2万篇）:
count    20000.000000
mean       343.684200
std        142.409536
min          9.000000
25%        247.000000
50%        350.000000
75%        441.000000
max       2367.000000
Name: token_count, dtype: float64

P90: 517
P95: 566
P99: 674

超过512 tokens的文章占比: 10.69%


In [71]:
cleaned_output_path = "../data/processed/dataset_expanded_clean.json"
df_clean_expanded.to_json(cleaned_output_path, orient="records", force_ascii=False, indent=1)
print(f"清洗后数据已保存到 {cleaned_output_path}，共 {len(df_clean_expanded)} 条")

清洗后数据已保存到 ../data/processed/dataset_expanded_clean.json，共 138489 条


## 5.分层抽样

In [74]:
import random
random.seed(1)

def token_len(r):
    full_text = f"{r.get('title') or ''}\n{r['abstract']}"
    return len(tokenizer.encode(full_text, truncation=False))

short_pool = [r for r in valid if token_len(r) < 261]
medium_pool = [r for r in valid if 261 <= token_len(r) < 440]
long_pool = [r for r in valid if token_len(r) >= 440]

print(f"短文本池(<261 tokens): {len(short_pool)} 篇")
print(f"中等文本池(261-440 tokens): {len(medium_pool)} 篇")
print(f"长文本池(>=440 tokens): {len(long_pool)} 篇")

sample_short = random.sample(short_pool, 8)
sample_medium = random.sample(medium_pool, 8)
sample_long = random.sample(long_pool, 8)

print("\n=== 短文本抽样（8篇，全文）===")
for r in sample_short:
    print(f"\n[{r['pmc_id']}] {r['title']}")
    print(r['abstract'])
    print("-" * 80)

print("\n\n=== 中等文本抽样（8篇，全文）===")
for r in sample_medium:
    print(f"\n[{r['pmc_id']}] {r['title']}")
    print(r['abstract'])
    print("-" * 80)

print("\n\n=== 长文本抽样（8篇，全文）===")
for r in sample_long:
    print(f"\n[{r['pmc_id']}] {r['title']}")
    print(r['abstract'])
    print("-" * 80)

短文本池(<261 tokens): 693 篇
中等文本池(261-440 tokens): 1386 篇
长文本池(>=440 tokens): 696 篇

=== 短文本抽样（8篇，全文）===

[PMC549212] In vitro bioassay as a predictor of in vivo response
Background There is a substantial discrepancy between  in vitro  and  in vivo  experiments. The purpose of the present work was development of a theoretical framework to enable improved prediction of  in vivo  response from  in vitro  bioassay results. Results For dose-response curve reaches a plateau  in vitro  we demonstrated that the  in vivo  response has only one maximum. For biphasic patterns of biological response  in vitro  both the bimodal and biphasic  in vivo  responses might be observed. Conclusion As the main result of this work we have demonstrated that  in vivo  responses might be predicted from dose-effect curves measured  in vitro .
--------------------------------------------------------------------------------

[PMC387267] Calcium Dynamics of Cortical Astrocytic Networks In Vivo
Large and long-lasting 

### 高频词分析

In [77]:
from collections import Counter

STOPWORDS = set("""
the a an of and in to with for was were is are this that on at by as from
have not been also could would should may might can will shall must do does
did has had but more than two one all both other after during there most
group however such when into among each who while over only its those
the a an
""".split())

words = []
for r in valid:
    text = (r.get("title") or "") + " " + r["abstract"]
    for w in text.split():
        w = w.strip('.,();:%').lower()
        if w and w not in STOPWORDS and len(w) > 2:
            words.append(w)

freq = Counter(words)
print("=== 高频词 Top20（去停用词）===")
for w, c in freq.most_common(20):
    print(f"  {w:20s} {c}")

=== 高频词 Top20（去停用词）===
  results              2707
  patients             2125
  background           2024
  study                1860
  these                1830
  cells                1600
  methods              1526
  data                 1508
  expression           1491
  between              1404
  gene                 1387
  which                1337
  genes                1296
  analysis             1283
  cell                 1262
  using                1258
  conclusions          1174
  protein              1130
  used                 1108
  health               1087


## 6. 分块预览

In [80]:
import json
import pandas as pd

with open("../data/processed/chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

chunks_df = pd.DataFrame(chunks)
print(f"总chunk数: {len(chunks_df)}")
print(f"涉及文献数: {chunks_df['doc_id'].nunique()}")

总chunk数: 170237
涉及文献数: 138489


### 样例1：智能分割（a分支）

In [83]:
multi_chunk_doc_ids = chunks_df[chunks_df["total_chunks"] > 1]["doc_id"].unique()
sample_doc_id = pd.Series(multi_chunk_doc_ids).sample(1, random_state=42).iloc[0]
sample_a = chunks_df[chunks_df["doc_id"] == sample_doc_id].sort_values("chunk_index")

print(f"\n样例1：智能分割（a分支）doc_id={sample_doc_id}，共{len(sample_a)}块")
for _, c in sample_a.iterrows():
    print(f"  chunk{c['chunk_index']} ({c['token_count']} tokens): {c['text'][:100]}...\n")



样例1：智能分割（a分支）doc_id=PMC2708127，共2块
  chunk0 (400 tokens): Atosiban versus betamimetics in the treatment of preterm labour in Germany: an economic evaluation. ...

  chunk1 (235 tokens): Compared to betamimetics, use of atosiban was associated with a significantly lower frequency of adv...



### 样例2：整体不分割（b分支）

In [86]:
sample_b = chunks_df[chunks_df["total_chunks"] == 1].sample(1, random_state=42).iloc[0]
print("\n样例2：整体不分割（b分支）")
print(f"doc_id: {sample_b['doc_id']}")
print(f"token_count: {sample_b['token_count']}")
print(f"text预览: {sample_b['text'][:200]}...")


样例2：整体不分割（b分支）
doc_id: PMC2908537
token_count: 418
text预览: Identification of the Rheumatoid Arthritis Shared Epitope Binding Site on Calreticulin. Background The rheumatoid arthritis (RA) shared epitope (SE), a major risk factor for severe disease, is a five ...


### 随机抽样5条

In [89]:
print(" 随机抽样5条chunk ")
for _, c in chunks_df.sample(5, random_state=1).iterrows():
    print(f"[{c['chunk_id']}] ({c['token_count']} tokens): {c['text'][:100]}...\n")


 随机抽样5条chunk 
[PMC2971393] (157 tokens): Poly[bis­[μ2-2-(1H-1,2,4-triazol-1-yl)acetato]zinc(II)]. In the title compound, [Zn(C 4 H 4 N 3 O 2 ...

[PMC1281289_chunk0] (382 tokens): Effect of Maternal Smoking on Breast Milk Interleukin-1α, β-Endorphin, and Leptin Concentrations. To...

[PMC2660282] (335 tokens): Resistance training with soy vs whey protein supplements in hyperlipidemic males. Background Most in...

[PMC1159039] (217 tokens): Simultaneous imaging of GFP, CFP and collagen in tumors in vivo using multiphoton microscopy. Backgr...

[PMC2268709_chunk0] (398 tokens): Forced expression of the cell cycle inhibitor p57Kip2 in cardiomyocytes attenuates ischemia-reperfus...



## 7. 质量验证

### 对生成的文本块进行质量抽样检查

In [93]:
# 1. 总块数，是否超过模型限制
print(" 1. 总块数与模型限制检查")
print(f"总chunk数: {len(chunks_df)}")

over_limit = chunks_df[chunks_df["token_count"] > 512]
print(f"超过512 tokens的chunk数: {len(over_limit)}")


 1. 总块数与模型限制检查
总chunk数: 170237
超过512 tokens的chunk数: 0


In [95]:
# 2. 文本质量

print("\n 2. 文本质量 ")

# ---- 2.1 是否包含标题 ----
import re
missing_title = chunks_df[chunks_df["source_title"].isna() | (chunks_df["source_title"].str.strip() == "")]
print(f"[标题] source_title缺失的chunk数: {len(missing_title)}")

# ---- 2.2 文本是否被不完整截断 ----
def looks_truncated_start(text):
    stripped = text.lstrip()
    if not stripped:
        return True
    return bool(re.match(r'^[.,;:)\]]', stripped)) or stripped[0].islower()

def looks_truncated_end(text):
    stripped = text.rstrip()
    if not stripped:
        return True
    return stripped[-1] not in ".!?\"'）)"

non_first_chunks = chunks_df[chunks_df["chunk_index"] > 0]
truncated_start = non_first_chunks[non_first_chunks["text"].apply(looks_truncated_start)]
print(f"[截断] 开头疑似截断的chunk数（非首块）: {len(truncated_start)} / {len(non_first_chunks)}")

not_last_chunks = chunks_df[chunks_df["chunk_index"] < chunks_df["total_chunks"] - 1]
truncated_end = not_last_chunks[not_last_chunks["text"].apply(looks_truncated_end)]
print(f"[截断] 结尾疑似截断的chunk数（非末块）: {len(truncated_end)} / {len(not_last_chunks)}")



 2. 文本质量 
[标题] source_title缺失的chunk数: 0
[截断] 开头疑似截断的chunk数（非首块）: 1438 / 31748
[截断] 结尾疑似截断的chunk数（非末块）: 880 / 31748


In [97]:
def starts_with_stray_punct(text):
    stripped = text.lstrip()
    return bool(re.match(r'^[.,;:)\]]', stripped))

def starts_lowercase_midsentence(text):
    stripped = text.lstrip()
    return bool(stripped) and stripped[0].islower() and not starts_with_stray_punct(text)

stray_punct = non_first_chunks[non_first_chunks["text"].apply(starts_with_stray_punct)]
midsentence = non_first_chunks[non_first_chunks["text"].apply(starts_lowercase_midsentence)]

print(f"[真异常] 开头是孤立标点的chunk数: {len(stray_punct)} / {len(non_first_chunks)}")
print(f"[正常现象] 因overlap从句中开始的chunk数: {len(midsentence)} / {len(non_first_chunks)}")

[真异常] 开头是孤立标点的chunk数: 9 / 31748
[正常现象] 因overlap从句中开始的chunk数: 1429 / 31748


### 对多块分割文献重点检查

In [100]:
# 1.总块数/超限（只看多块文献部分）
multi_chunk_docs = chunks_df[chunks_df["total_chunks"] > 1]
print(f"多块分割的文献数: {multi_chunk_docs['doc_id'].nunique()}")
multi_over_limit = multi_chunk_docs[multi_chunk_docs["token_count"] > 512]
print(f"[多块-超限] 多块文献中超过512 tokens的chunk数: {len(multi_over_limit)} / {len(multi_chunk_docs)}")


多块分割的文献数: 30843
[多块-超限] 多块文献中超过512 tokens的chunk数: 0 / 62591


In [102]:
# 2 文本质量（只看多块文献部分）
multi_missing_title = multi_chunk_docs[multi_chunk_docs["source_title"].isna() | (multi_chunk_docs["source_title"].str.strip() == "")]
print(f"[多块-标题] 多块文献中标题缺失的chunk数: {len(multi_missing_title)}")

multi_non_first = multi_chunk_docs[multi_chunk_docs["chunk_index"] > 0]
multi_truncated_start = multi_non_first[multi_non_first["text"].apply(looks_truncated_start)]
print(f"[多块-截断] 多块文献中开头疑似截断的chunk数: {len(multi_truncated_start)} / {len(multi_non_first)}")

[多块-标题] 多块文献中标题缺失的chunk数: 0
[多块-截断] 多块文献中开头疑似截断的chunk数: 1438 / 31748


In [104]:
# 3. 结构一致性
def check_doc_consistency(group):
    n = len(group)
    expected_total = group["total_chunks"].iloc[0]
    indices = sorted(group["chunk_index"].tolist())
    return pd.Series({
        "is_continuous": indices == list(range(n)),
        "total_matches": (n == expected_total),
    })

consistency_check = multi_chunk_docs.groupby("doc_id").apply(check_doc_consistency, include_groups=False)
print(f"[多块-结构] chunk_index不连续的文献数: {(~consistency_check['is_continuous']).sum()}")
print(f"[多块-结构] total_chunks不符的文献数: {(~consistency_check['total_matches']).sum()}")

[多块-结构] chunk_index不连续的文献数: 0
[多块-结构] total_chunks不符的文献数: 0


In [132]:
# 4. 重叠部分检查
def char_overlap_length(text_a, text_b, max_check=500):
    tail = text_a[-max_check:]
    for overlap_len in range(min(len(tail), len(text_b)), 0, -1):
        if tail[-overlap_len:] == text_b[:overlap_len]:
            return overlap_len
    return 0

overlap_results = []
for doc_id, group in multi_chunk_docs.groupby("doc_id"):
    group_sorted = group.sort_values("chunk_index").reset_index(drop=True)
    for i in range(len(group_sorted) - 1):
        overlap_chars = char_overlap_length(group_sorted.loc[i, "text"], group_sorted.loc[i + 1, "text"])
        overlap_results.append({"doc_id": doc_id, "chunk_pair": f"{i}->{i+1}", "overlap_chars": overlap_chars})

overlap_df = pd.DataFrame(overlap_results)
print(f"[多块-重叠] 共检查 {len(overlap_df)} 对相邻chunk")
print(f"[多块-重叠] 零重叠的chunk对数: {(overlap_df['overlap_chars'] == 0).sum()}")
print(f"[多块-重叠] 重叠字符数分布:\n{overlap_df['overlap_chars'].describe()}")

[多块-重叠] 共检查 31748 对相邻chunk
[多块-重叠] 零重叠的chunk对数: 2383
[多块-重叠] 重叠字符数分布:
count    31748.000000
mean       210.952910
std         91.697471
min          0.000000
25%        162.000000
50%        221.000000
75%        274.000000
max        494.000000
Name: overlap_chars, dtype: float64


In [134]:
zero_pairs = overlap_df[overlap_df["overlap_chars"] == 0].sample(3, random_state=1)

for _, row in zero_pairs.iterrows():
    doc_id = row["doc_id"]
    i0, i1 = row["chunk_pair"].split("->")
    group = multi_chunk_docs[multi_chunk_docs["doc_id"] == doc_id].sort_values("chunk_index").reset_index(drop=True)
    text_a = group.loc[int(i0), "text"]
    text_b = group.loc[int(i1), "text"]
    print(f"doc_id={doc_id}, {row['chunk_pair']} ")
    print(f"chunk{i0}结尾: ...{text_a[-100:]}")
    print(f"chunk{i1}开头: {text_b[:100]}...")
    print()

doc_id=PMC1971700, 0->1 
chunk0结尾: ...ith liver metastasis and to 94 +/- 0.05 mumol l-1 liver in cancer patients without liver malignancy.
chunk1开头: In vitro investigation of HBP density revealed the malignant liver tissue to have a significantly (P...

doc_id=PMC2874535, 1->2 
chunk1结尾: ...PWV differences = -0.12(1.0), 0.8(1.7), and 0.65(1.6) m/s for TT-XC, TT-QA, and XC-QA, respectively.
chunk2开头: Finally, in the group of diastolic heart failure patient, PWV was significantly higher (6.3 ± 1.9 m/...

doc_id=PMC2817686, 0->1 
chunk0结尾: ... CDH1 ,  FOXG1 ,  IGSF3 ,  ISL1 ,  MALL ,  PLAU ,  RAB25 ,  S100P ,  SLCO4A1 ,  STMN1 , and  TGM2 ).
chunk1开头: The combined PCA and gene pathway analyses suggested that these genes were related to cell adhesion,...



In [136]:
# 抽查几个零重叠样例，看看被跳过的句子有多长
zero_overlap_sample = overlap_df[overlap_df["overlap_chars"] == 0].sample(5, random_state=2)

for _, row in zero_overlap_sample.iterrows():
    doc_id = row["doc_id"]
    i0, i1 = row["chunk_pair"].split("->")
    group = multi_chunk_docs[multi_chunk_docs["doc_id"] == doc_id].sort_values("chunk_index").reset_index(drop=True)
    chunk0_tokens = group.loc[int(i0), "token_count"]
    chunk1_tokens = group.loc[int(i1), "token_count"]
    print(f"doc_id={doc_id}, chunk{i0}({chunk0_tokens} tokens) -> chunk{i1}({chunk1_tokens} tokens)")

doc_id=PMC2653526, chunk0(340 tokens) -> chunk1(191 tokens)
doc_id=PMC2360121, chunk0(398 tokens) -> chunk1(111 tokens)
doc_id=PMC2820547, chunk0(48 tokens) -> chunk1(341 tokens)
doc_id=PMC2361699, chunk0(375 tokens) -> chunk1(93 tokens)
doc_id=PMC2794027, chunk0(395 tokens) -> chunk1(153 tokens)


In [138]:
# 假设一：末句token数超出80-token重叠预算，分割器放弃重叠

def count_str_tokens(text):
    return len(tokenizer.encode(text, truncation=False))

def get_last_sentence_tokens(text):
    for sep in ["\n\n", "\n", ". "]:
        if sep in text:
            last_piece = text.split(sep)[-1]
            return count_str_tokens(last_piece)
    return count_str_tokens(text)

zero_overlap_check = overlap_df[overlap_df["overlap_chars"] == 0].sample(10, random_state=3)
for _, row in zero_overlap_check.iterrows():
    doc_id = row["doc_id"]
    i0, i1 = row["chunk_pair"].split("->")
    group = multi_chunk_docs[multi_chunk_docs["doc_id"] == doc_id].sort_values("chunk_index").reset_index(drop=True)
    text_a = group.loc[int(i0), "text"]
    last_sent_tokens = get_last_sentence_tokens(text_a)
    print(f"doc_id={doc_id}: 末句token数≈{last_sent_tokens}")

doc_id=PMC2365957: 末句token数≈117
doc_id=PMC2265763: 末句token数≈88
doc_id=PMC2706978: 末句token数≈250
doc_id=PMC2854156: 末句token数≈56
doc_id=PMC1592340: 末句token数≈3
doc_id=PMC2267444: 末句token数≈113
doc_id=PMC2248176: 末句token数≈86
doc_id=PMC2907589: 末句token数≈100
doc_id=PMC2920244: 末句token数≈88
doc_id=PMC2265305: 末句token数≈109


In [140]:
# 检测方法过于严格（要求逐字符精确匹配）导致误判 

def sentence_reappears(text_a, text_b, check_chars=200):
    # 取chunk0最后一句（去除首尾空白，标准化多余空格）
    for sep in ["\n\n", "\n", ". "]:
        if sep in text_a:
            last_sent = text_a.split(sep)[-1].strip()
            break
    else:
        last_sent = text_a.strip()

    last_sent_norm = " ".join(last_sent.split())  # 标准化空白
    text_b_norm = " ".join(text_b[:check_chars].split())

    return last_sent_norm in text_b_norm, last_sent_norm[:50]

zero_overlap_check2 = overlap_df[overlap_df["overlap_chars"] == 0].sample(10, random_state=3)
for _, row in zero_overlap_check2.iterrows():
    doc_id = row["doc_id"]
    i0, i1 = row["chunk_pair"].split("->")
    group = multi_chunk_docs[multi_chunk_docs["doc_id"] == doc_id].sort_values("chunk_index").reset_index(drop=True)
    text_a = group.loc[int(i0), "text"]
    text_b = group.loc[int(i1), "text"]
    found, preview = sentence_reappears(text_a, text_b)
    print(f"doc_id={doc_id}: 末句是否重新出现在chunk1开头附近: {found} | 末句预览: {preview}...")

doc_id=PMC2365957: 末句是否重新出现在chunk1开头附近: False | 末句预览: At week 48, intent-to-treat: missing/discontinuati...
doc_id=PMC2265763: 末句是否重新出现在chunk1开头附近: False | 末句预览: When maternal FTO, controlling for offspring FTO, ...
doc_id=PMC2706978: 末句是否重新出现在chunk1开头附近: False | 末句预览: Results . The predominant adverse event was transi...
doc_id=PMC2854156: 末句是否重新出现在chunk1开头附近: False | 末句预览: to apical membranes; to an ERBIN family protein (L...
doc_id=PMC1592340: 末句是否重新出现在chunk1开头附近: False | 末句预览: Background...
doc_id=PMC2267444: 末句是否重新出现在chunk1开头附近: False | 末句预览: Further, NCX-4040 significantly enhanced the sensi...
doc_id=PMC2248176: 末句是否重新出现在chunk1开头附近: False | 末句预览: Se encontró, en este caso, que la similitud de las...
doc_id=PMC2907589: 末句是否重新出现在chunk1开头附近: False | 末句预览: However, comparisons between pure species indicate...
doc_id=PMC2920244: 末句是否重新出现在chunk1开头附近: False | 末句预览: Normoglycemic group (n = 24; W = 19, M = 5) had no...
doc_id=PMC2265305: 末句是否重新出现在chunk1开头附近: False | 末句预览: In males, we o

In [142]:
print(f"共检查 {len(overlap_df)} 对相邻chunk")
print(f"零重叠的chunk对数: {(overlap_df['overlap_chars'] == 0).sum()}")
print(f"零重叠占比: {(overlap_df['overlap_chars'] == 0).mean():.2%}")

共检查 31748 对相邻chunk
零重叠的chunk对数: 2383
零重叠占比: 7.51%


In [144]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

class InstrumentedSplitter(RecursiveCharacterTextSplitter):
    def _merge_splits(self, splits, separator):
        separator_len = self._length_function(separator)
        docs = []
        current_doc = []
        total = 0
        for d in splits:
            len_ = self._length_function(d)
            if (total + len_ + (separator_len if len(current_doc) > 0 else 0) > self._chunk_size):
                if len(current_doc) > 0:
                    doc = self._join_docs(current_doc, separator)
                    if doc is not None:
                        docs.append(doc)
                    total_before_pop = total
                    while total > self._chunk_overlap or (
                        total + len_ + (separator_len if len(current_doc) > 0 else 0) > self._chunk_size
                        and total > 0
                    ):
                        total -= self._length_function(current_doc[0]) + (
                            separator_len if len(current_doc) > 1 else 0
                        )
                        current_doc = current_doc[1:]
                    print(f"[边界] finalize时total={total_before_pop}, 触发split的len_={len_}")
                    print(f"       弹出后重叠种子: total={total}, 内容预览={''.join(current_doc)[:80]!r}")
                    print()
            current_doc.append(d)
            total += len_ + (separator_len if len(current_doc) > 1 else 0)
        doc = self._join_docs(current_doc, separator)
        if doc is not None:
            docs.append(doc)
        return docs

inst_splitter = InstrumentedSplitter(
    chunk_size=450, chunk_overlap=80,
    length_function=count_tokens,
    separators=["\n\n", "\n", ". ", " ", ""],
    keep_separator="end",
)

for doc_id in ["PMC2858031", "PMC2939607", "PMC2821401"]:
    row = df_clean_expanded[df_clean_expanded["pmc_id"] == doc_id].iloc[0]
    print(f"=== {doc_id} ===")
    _ = inst_splitter.split_text(build_full_text(row))
    print()

=== PMC2858031 ===
[边界] finalize时total=450, 触发split的len_=13
       弹出后重叠种子: total=64, 内容预览='Pre-exposure of IB3-1 cells to  P. aeruginosa  PAO1 significantly increased  S. '


=== PMC2939607 ===
[边界] finalize时total=425, 触发split的len_=60
       弹出后重叠种子: total=61, 内容预览='However, the fastest healing was observed in mice injected with bicistronic AAV-'


=== PMC2821401 ===
[边界] finalize时total=448, 触发split的len_=11
       弹出后重叠种子: total=65, 内容预览="Constructs containing both SA domains and Kaiso's aminoterminal BTB/POZ domain t"




In [146]:
row = df_clean_expanded[df_clean_expanded["pmc_id"] == "PMC2858031"].iloc[0]
fresh_chunks = inst_splitter.split_text(build_full_text(row))
print(f"重新生成的chunk数: {len(fresh_chunks)}")
for i, t in enumerate(fresh_chunks):
    print(f"chunk{i} ({count_tokens(t)} tokens): {t[:80]!r}")
print()

saved = chunks_df[chunks_df["doc_id"] == "PMC2858031"].sort_values("chunk_index")
print(f"chunks.json里保存的chunk数: {len(saved)}")
for _, c in saved.iterrows():
    print(f"chunk{c['chunk_index']} ({c['token_count']} tokens): {c['text'][:80]!r}")

[边界] finalize时total=450, 触发split的len_=13
       弹出后重叠种子: total=64, 内容预览='Pre-exposure of IB3-1 cells to  P. aeruginosa  PAO1 significantly increased  S. '

重新生成的chunk数: 2
chunk0 (378 tokens): 'Adhesion to and biofilm formation on IB3-1 bronchial cells by Stenotrophomonas m'
chunk1 (149 tokens): 'Pre-exposure of IB3-1 cells to  P. aeruginosa  PAO1 significantly increased  S. '

chunks.json里保存的chunk数: 2
chunk0 (378 tokens): 'Adhesion to and biofilm formation on IB3-1 bronchial cells by Stenotrophomonas m'
chunk1 (149 tokens): 'Pre-exposure of IB3-1 cells to  P. aeruginosa  PAO1 significantly increased  S. '


In [148]:
row = df_clean_expanded[df_clean_expanded["pmc_id"] == "PMC2858031"].iloc[0]
fresh_chunks = inst_splitter.split_text(build_full_text(row))

chunk0_text, chunk1_text = fresh_chunks[0], fresh_chunks[1]

print("chunk0结尾150字符:")
print(repr(chunk0_text[-150:]))
print()
print("chunk1开头150字符:")
print(repr(chunk1_text[:150]))
print()

# 直接在新鲜文本上跑一遍原来的重叠检测函数
print("char_overlap_length结果:", char_overlap_length(chunk0_text, chunk1_text))

[边界] finalize时total=450, 触发split的len_=13
       弹出后重叠种子: total=64, 内容预览='Pre-exposure of IB3-1 cells to  P. aeruginosa  PAO1 significantly increased  S. '

chunk0结尾150字符:
'lls to  P. aeruginosa  PAO1 significantly increased  S. maltophilia  adhesiveness. Further, the presence of  S. maltophilia  negatively influenced  P.'

chunk1开头150字符:
'Pre-exposure of IB3-1 cells to  P. aeruginosa  PAO1 significantly increased  S. maltophilia  adhesiveness. Further, the presence of  S. maltophilia  n'

char_overlap_length结果: 174


In [150]:
# 1. 确认特殊token是否被重复叠加
print("带特殊token:", tokenizer.encode("Cells are healthy."))
print("不带特殊token:", tokenizer.encode("Cells are healthy.", add_special_tokens=False))

# 2. 用真实文档验证：逐句单独计数求和 vs 整体拼接后一次性计数
row = df_clean_expanded[df_clean_expanded["pmc_id"] == "PMC2858031"].iloc[0]
full_text = build_full_text(row)

rough_sentences = full_text.split(". ")
rough_sentences = [s + ". " for s in rough_sentences[:-1]] + [rough_sentences[-1]]

sum_individual = sum(count_tokens(s) for s in rough_sentences)
joined_count = count_tokens("".join(rough_sentences))

print(f"逐句单独计数求和: {sum_individual}")
print(f"整体拼接后一次性计数: {joined_count}")
print(f"差值: {sum_individual - joined_count}（共{len(rough_sentences)}句）")

带特殊token: [101, 4442, 2024, 7965, 1012, 102]
不带特殊token: [4442, 2024, 7965, 1012]
逐句单独计数求和: 523
整体拼接后一次性计数: 479
差值: 44（共23句）


In [152]:
def char_overlap_length_v2(text_a, text_b, max_check=500):
    tail = text_a[-max_check:]
    for overlap_len in range(min(len(tail), len(text_b)), 0, -1):
        if tail[-overlap_len:] == text_b[:overlap_len]:
            return overlap_len
    return 0

print("原版(150字符窗口):", char_overlap_length(chunk0_text, chunk1_text))
print("放大窗口(500字符):", char_overlap_length_v2(chunk0_text, chunk1_text))

原版(150字符窗口): 174
放大窗口(500字符): 174


In [155]:
# 用更大的窗口重跑一次，看max是否还会继续增大
overlap_df_check = []
for doc_id, group in multi_chunk_docs.groupby("doc_id"):
    group_sorted = group.sort_values("chunk_index").reset_index(drop=True)
    for i in range(len(group_sorted) - 1):
        oc = char_overlap_length(group_sorted.loc[i, "text"], group_sorted.loc[i+1, "text"], max_check=800)
        overlap_df_check.append(oc)

import pandas as pd
print(pd.Series(overlap_df_check).describe())

count    31748.000000
mean       210.952910
std         91.697471
min          0.000000
25%        162.000000
50%        221.000000
75%        274.000000
max        494.000000
dtype: float64
